# 04. Model Training and Evaluation

This notebook trains and compares the AquaSense AI intelligence layers:

1. breach prediction classifier;
2. pollutant forecasting regressors;
3. anomaly detection layer;
4. model selection and artifact export.

The notebook assumes `notebooks/03_preprocessing.ipynb` has already produced:

```text
data/processed/preprocessed_model_matrix.csv
data/processed/preprocessing_feature_manifest.csv
data/processed/temporal_split_summary.csv
data/processed/preprocessing_audit.json
```

No random train/test split is used. The notebook keeps the existing chronological `train`, `validation`, and `test` splits.


## 1. Setup


In [3]:
from pathlib import Path
import json
import pickle
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import (
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    IsolationForest,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.exceptions import ConvergenceWarning
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_curve,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

import joblib

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "services" / "ml" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

OUTPUTS = {
    "classification_results": PROCESSED / "model_classification_results.csv",
    "forecast_results": PROCESSED / "model_forecast_results.csv",
    "anomaly_results": PROCESSED / "model_anomaly_results.csv",
    "feature_importance": PROCESSED / "model_feature_importance.csv",
    "model_card": PROCESSED / "model_card_summary.json",
    "calibration_results": PROCESSED / "model_calibration_results.csv",
    "lead_time_results": PROCESSED / "model_lead_time_results.csv",
    "incident_holdout_results": PROCESSED / "model_incident_holdout_results.csv",
    "walk_forward_results": PROCESSED / "model_walk_forward_results.csv",
    "model_selection": PROCESSED / "model_selection_summary.json",
    "best_model_h5": MODEL_DIR / "aquasense_best_models.h5",
    "best_model_joblib": MODEL_DIR / "aquasense_best_models.joblib",
}

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
OUTPUTS


{'classification_results': WindowsPath('c:/Users/nwagb/AQUASENSE-AI/data/processed/model_classification_results.csv'),
 'forecast_results': WindowsPath('c:/Users/nwagb/AQUASENSE-AI/data/processed/model_forecast_results.csv'),
 'anomaly_results': WindowsPath('c:/Users/nwagb/AQUASENSE-AI/data/processed/model_anomaly_results.csv'),
 'feature_importance': WindowsPath('c:/Users/nwagb/AQUASENSE-AI/data/processed/model_feature_importance.csv'),
 'model_selection': WindowsPath('c:/Users/nwagb/AQUASENSE-AI/data/processed/model_selection_summary.json'),
 'best_model_h5': WindowsPath('c:/Users/nwagb/AQUASENSE-AI/services/ml/models/aquasense_best_models.h5'),
 'best_model_joblib': WindowsPath('c:/Users/nwagb/AQUASENSE-AI/services/ml/models/aquasense_best_models.joblib')}

## 2. Optional Model Libraries


In [4]:
# Set this to True if you are running the notebook on a fresh environment
# and want to install the optional comparison libraries from inside Jupyter.
INSTALL_OPTIONAL_MODEL_LIBRARIES = False

if INSTALL_OPTIONAL_MODEL_LIBRARIES:
    import sys
    import subprocess

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "lightgbm",
        "xgboost",
        "catboost",
        "h5py",
    ])


In [5]:
optional_libraries = {}

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    optional_libraries["lightgbm"] = True
except Exception:
    optional_libraries["lightgbm"] = False

try:
    from xgboost import XGBClassifier, XGBRegressor
    optional_libraries["xgboost"] = True
except Exception:
    optional_libraries["xgboost"] = False

try:
    from catboost import CatBoostClassifier, CatBoostRegressor
    optional_libraries["catboost"] = True
except Exception:
    optional_libraries["catboost"] = False

try:
    import h5py
    optional_libraries["h5py"] = True
except Exception:
    h5py = None
    optional_libraries["h5py"] = False

optional_libraries


{'lightgbm': True, 'xgboost': True, 'catboost': True, 'h5py': True}

## 3. Load Preprocessed Matrix


In [6]:
required_paths = {
    "model_matrix": PROCESSED / "preprocessed_model_matrix.csv",
    "feature_manifest": PROCESSED / "preprocessing_feature_manifest.csv",
    "split_summary": PROCESSED / "temporal_split_summary.csv",
    "preprocessing_audit": PROCESSED / "preprocessing_audit.json",
}

missing = {name: str(path) for name, path in required_paths.items() if not path.exists()}
if missing:
    raise FileNotFoundError(
        "Missing preprocessing outputs. Run notebooks/03_preprocessing.ipynb first. "
        f"Missing: {missing}"
    )

manifest = pd.read_csv(required_paths["feature_manifest"])
split_summary = pd.read_csv(required_paths["split_summary"])
with open(required_paths["preprocessing_audit"], encoding="utf-8") as f:
    preprocessing_audit = json.load(f)

feature_cols = manifest.loc[manifest["keep_for_model"], "imputed_feature"].tolist()
target_cols = preprocessing_audit["target_columns"]
forecast_target_cols = preprocessing_audit["forecast_target_columns"]

required_cols = [
    "timestamp", "facility_id", "split", "breach_now_bool",
    "sensor_status", "compliance_status", "main_risk_driver", "event_type",
    "breach_episode_id", "incident_group_id", "minutes_to_next_breach",
] + target_cols + forecast_target_cols + feature_cols

available_cols = pd.read_csv(required_paths["model_matrix"], nrows=0).columns.tolist()
usecols = [col for col in required_cols if col in available_cols]
model_df = pd.read_csv(required_paths["model_matrix"], usecols=usecols, parse_dates=["timestamp"])

for col in feature_cols:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

print(model_df.shape)
split_summary


(8604, 436)


,split,rows,start,end,breach_next_30min_rate,current_breach_rate
0,train,6022,2026-05-01 02:00:00,2026-05-21 23:45:00,0.88,0.432
1,validation,1291,2026-05-21 23:50:00,2026-05-26 11:20:00,0.93,0.542
2,test,1291,2026-05-26 11:25:00,2026-05-30 22:55:00,0.00,0.000


## 4. Final Leakage and Split Guards


In [7]:
forbidden_tokens = [
    "target_", "breach_next_", "lab_", "event_type", "recommended",
    "current_breach_reason", "compliance_status", "main_risk_driver",
]
leakage_hits = [col for col in feature_cols if any(token in col for token in forbidden_tokens)]
if leakage_hits:
    raise AssertionError(f"Leakage-like feature columns found: {leakage_hits[:20]}")

expected_splits = {"train", "validation", "test"}
if set(model_df["split"].unique()) != expected_splits:
    raise AssertionError(f"Expected splits {expected_splits}, found {set(model_df['split'].unique())}")

if not model_df["timestamp"].is_monotonic_increasing:
    raise AssertionError("Model matrix must remain globally timestamp-sorted")

split_ranges = (
    model_df.groupby("split", sort=False)
    .agg(rows=("timestamp", "size"), start=("timestamp", "min"), end=("timestamp", "max"))
    .reset_index()
)

target_rates = (
    model_df.groupby("split", sort=False)[target_cols]
    .mean()
    .mul(100)
    .round(3)
)

split_ranges, target_rates


(        split  rows               start                 end
 0       train  6022 2026-05-01 02:00:00 2026-05-21 23:45:00
 1  validation  1291 2026-05-21 23:50:00 2026-05-26 11:20:00
 2        test  1291 2026-05-26 11:25:00 2026-05-30 22:55:00,
             target_breach_next_15min  target_breach_next_30min  target_breach_next_60min
 split                                                                                   
 train                          0.631                      0.88                     1.378
 validation                     0.697                      0.93                     1.394
 test                           0.000                      0.00                     0.000)

## 5. Train / Validation / Test Matrices


In [8]:
target = "target_breach_next_30min"

train_df = model_df.loc[model_df["split"].eq("train")].copy()
valid_df = model_df.loc[model_df["split"].eq("validation")].copy()
test_df = model_df.loc[model_df["split"].eq("test")].copy()

X_train = train_df[feature_cols]
X_valid = valid_df[feature_cols]
X_test = test_df[feature_cols]

y_train = train_df[target].astype(int)
y_valid = valid_df[target].astype(int)
y_test = test_df[target].astype(int)

{
    "target": target,
    "X_train": X_train.shape,
    "X_valid": X_valid.shape,
    "X_test": X_test.shape,
    "train_positive_rate_pct": round(y_train.mean() * 100, 3),
    "valid_positive_rate_pct": round(y_valid.mean() * 100, 3),
    "test_positive_rate_pct": round(y_test.mean() * 100, 3),
}


{'target': 'target_breach_next_30min',
 'X_train': (6022, 420),
 'X_valid': (1291, 420),
 'X_test': (1291, 420),
 'train_positive_rate_pct': np.float64(0.88),
 'valid_positive_rate_pct': np.float64(0.93),
 'test_positive_rate_pct': np.float64(0.0)}

## 6. Metric Helpers


In [9]:
def safe_roc_auc(y_true, y_score):
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)


def safe_average_precision(y_true, y_score):
    if pd.Series(y_true).sum() == 0:
        return np.nan
    return average_precision_score(y_true, y_score)


def false_alarms_per_day(frame, y_true, y_pred):
    timestamps = pd.to_datetime(frame["timestamp"])
    days = max((timestamps.max() - timestamps.min()).total_seconds() / 86400, 1 / 24)
    false_alarms = int(((np.asarray(y_pred) == 1) & (np.asarray(y_true) == 0)).sum())
    return false_alarms / days


def evaluate_classifier(model_name, split_name, frame, y_true, y_score, threshold):
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "split": split_name,
        "threshold": threshold,
        "rows": len(y_true),
        "positive_rate_pct": round(float(np.mean(y_true) * 100), 4),
        "predicted_positive_rate_pct": round(float(np.mean(y_pred) * 100), 4),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred) if pd.Series(y_true).nunique() > 1 else np.nan,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": safe_roc_auc(y_true, y_score),
        "pr_auc": safe_average_precision(y_true, y_score),
        "false_alarms_per_day": false_alarms_per_day(frame, y_true, y_pred),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }


def threshold_table(model_name, frame, y_true, y_score, thresholds=(0.35, 0.50, 0.70, 0.90)):
    return pd.DataFrame([
        evaluate_classifier(model_name, "validation", frame, y_true, y_score, threshold)
        for threshold in thresholds
    ])


def choose_threshold(y_true, y_score, min_recall=0.85):
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    rows = []
    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        rows.append({"threshold": float(t), "precision": float(p), "recall": float(r), "f1": float(2 * p * r / (p + r)) if (p + r) else 0.0})
    table = pd.DataFrame(rows)
    if table.empty:
        return 0.5, table
    feasible = table.loc[table["recall"].ge(min_recall)]
    if len(feasible):
        best = feasible.sort_values(["f1", "precision"], ascending=False).iloc[0]
    else:
        best = table.sort_values(["f1", "recall"], ascending=False).iloc[0]
    return float(best["threshold"]), table


def regression_metrics(model_name, target_name, split_name, y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    denom = np.where(np.abs(y_true) < 1e-9, np.nan, np.abs(y_true))
    mape = np.nanmean(np.abs((y_true - y_pred) / denom)) * 100
    return {
        "model": model_name,
        "target": target_name,
        "split": split_name,
        "rows": len(y_true),
        "mae": mae,
        "rmse": rmse,
        "mape_pct": mape,
        "r2": r2_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan,
    }



def choose_operational_threshold(frame, y_true, y_score, min_recall=0.85, max_false_alarms_per_day=1.0):
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    rows = []
    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        y_pred = (np.asarray(y_score) >= t).astype(int)
        fad = false_alarms_per_day(frame, y_true, y_pred)
        f1 = float(2 * p * r / (p + r)) if (p + r) else 0.0
        rows.append({
            "threshold": float(t),
            "precision": float(p),
            "recall": float(r),
            "f1": f1,
            "false_alarms_per_day": float(fad),
        })
    table = pd.DataFrame(rows)
    if table.empty:
        return 0.5, table

    feasible = table.loc[
        table["recall"].ge(min_recall)
        & table["false_alarms_per_day"].le(max_false_alarms_per_day)
    ]
    if len(feasible):
        best = feasible.sort_values(["f1", "precision", "threshold"], ascending=[False, False, False]).iloc[0]
    else:
        table["operational_score"] = (
            4.0 * table["recall"]
            + 1.5 * table["precision"]
            - 0.15 * table["false_alarms_per_day"]
        )
        best = table.sort_values(["operational_score", "f1"], ascending=False).iloc[0]
    return float(best["threshold"]), table


def calibration_table(model_name, split_name, y_true, y_score, n_bins=10):
    frame = pd.DataFrame({"y_true": np.asarray(y_true), "score": np.asarray(y_score)})
    frame["bin"] = pd.cut(frame["score"], bins=np.linspace(0, 1, n_bins + 1), include_lowest=True)
    rows = (
        frame.groupby("bin", observed=False)
        .agg(rows=("y_true", "size"), mean_score=("score", "mean"), observed_rate=("y_true", "mean"))
        .reset_index()
    )
    rows["model"] = model_name
    rows["split"] = split_name
    rows["brier_score"] = brier_score_loss(y_true, y_score)
    return rows[["model", "split", "bin", "rows", "mean_score", "observed_rate", "brier_score"]]


def breach_episode_lead_times(frame, score, threshold, max_lookback_minutes=60):
    if "breach_episode_id" not in frame.columns:
        return pd.DataFrame(columns=[
            "breach_episode_id", "breach_start", "caught_before_breach",
            "first_alert_time", "lead_time_minutes", "alerts_before_breach"
        ])

    temp = frame[["timestamp", "breach_now_bool", "breach_episode_id"]].copy()
    temp["score"] = np.asarray(score)
    temp["alert"] = temp["score"].ge(threshold)
    temp["timestamp"] = pd.to_datetime(temp["timestamp"])

    breach_rows = temp.loc[temp["breach_episode_id"].ne("no_current_breach")].copy()
    if breach_rows.empty:
        return pd.DataFrame(columns=[
            "breach_episode_id", "breach_start", "caught_before_breach",
            "first_alert_time", "lead_time_minutes", "alerts_before_breach"
        ])

    rows = []
    for episode_id, episode in breach_rows.groupby("breach_episode_id", sort=False):
        breach_start = episode["timestamp"].min()
        lookback_start = breach_start - pd.Timedelta(minutes=max_lookback_minutes)
        warning_window = temp.loc[
            temp["timestamp"].between(lookback_start, breach_start, inclusive="left")
        ]
        alerts = warning_window.loc[warning_window["alert"]]
        if len(alerts):
            first_alert = alerts["timestamp"].min()
            lead = (breach_start - first_alert).total_seconds() / 60
        else:
            first_alert = pd.NaT
            lead = np.nan
        rows.append({
            "breach_episode_id": episode_id,
            "breach_start": breach_start,
            "caught_before_breach": bool(len(alerts)),
            "first_alert_time": first_alert,
            "lead_time_minutes": lead,
            "alerts_before_breach": int(len(alerts)),
        })
    return pd.DataFrame(rows)


## 7. Candidate Breach Classifiers


In [10]:
classification_models = {
    "dummy_prior": DummyClassifier(strategy="prior"),
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=350,
        max_depth=None,
        min_samples_leaf=3,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "extra_trees": ExtraTreesClassifier(
        n_estimators=450,
        min_samples_leaf=2,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=350,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=0.05,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "mlp_neural_baseline": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=(96, 48),
            alpha=1e-3,
            learning_rate_init=1e-3,
            max_iter=500,
            early_stopping=True,
            random_state=RANDOM_STATE,
        )),
    ]),
}

if optional_libraries["lightgbm"]:
    classification_models["lightgbm"] = LGBMClassifier(
        objective="binary",
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.9,
        colsample_bytree=0.85,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )

if optional_libraries["xgboost"]:
    scale_pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
    classification_models["xgboost"] = XGBClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.85,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

if optional_libraries["catboost"]:
    classification_models["catboost"] = CatBoostClassifier(
        iterations=500,
        learning_rate=0.03,
        depth=5,
        loss_function="Logloss",
        eval_metric="PRAUC",
        auto_class_weights="Balanced",
        random_seed=RANDOM_STATE,
        verbose=False,
    )

list(classification_models)


['dummy_prior',
 'logistic_regression',
 'random_forest',
 'extra_trees',
 'hist_gradient_boosting',
 'mlp_neural_baseline',
 'lightgbm',
 'xgboost',
 'catboost']

## 8. Train and Compare Breach Classifiers


In [11]:
classification_rows = []
threshold_rows = []
classification_fitted = {}
classification_scores = {}
classification_thresholds = {}

for model_name, estimator in classification_models.items():
    start = time.perf_counter()
    model = clone(estimator)
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start

    if hasattr(model, "predict_proba"):
        valid_score = model.predict_proba(X_valid)[:, 1]
        test_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        valid_raw = model.decision_function(X_valid)
        test_raw = model.decision_function(X_test)
        valid_score = (valid_raw - valid_raw.min()) / max(valid_raw.max() - valid_raw.min(), 1e-9)
        test_score = (test_raw - valid_raw.min()) / max(valid_raw.max() - valid_raw.min(), 1e-9)
    else:
        valid_score = model.predict(X_valid)
        test_score = model.predict(X_test)

    best_threshold, pr_table = choose_operational_threshold(valid_df, y_valid, valid_score, min_recall=0.85, max_false_alarms_per_day=1.0)
    classification_thresholds[model_name] = best_threshold
    threshold_rows.append(threshold_table(model_name, valid_df, y_valid, valid_score).assign(chosen_threshold=best_threshold))

    valid_metrics = evaluate_classifier(model_name, "validation", valid_df, y_valid, valid_score, best_threshold)
    test_metrics = evaluate_classifier(model_name, "test", test_df, y_test, test_score, best_threshold)
    valid_metrics["fit_seconds"] = fit_seconds
    test_metrics["fit_seconds"] = fit_seconds
    classification_rows.extend([valid_metrics, test_metrics])

    classification_fitted[model_name] = model
    classification_scores[model_name] = {"validation": valid_score, "test": test_score}

classification_results = pd.DataFrame(classification_rows).sort_values(
    ["split", "pr_auc", "recall", "false_alarms_per_day"],
    ascending=[True, False, False, True],
)
threshold_comparison = pd.concat(threshold_rows, ignore_index=True)

classification_results


,model,split,threshold,rows,positive_rate_pct,predicted_positive_rate_pct,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,false_alarms_per_day,tp,fp,tn,fn,fit_seconds
5,random_forest,test,0.101647,1291,0.0000,0.0000,1.000000,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.000000,0,0,1291,0,0.920940
9,hist_gradient_boosting,test,0.005623,1291,0.0000,0.0000,1.000000,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.000000,0,0,1291,0,4.277012
15,xgboost,test,0.025001,1291,0.0000,0.0000,1.000000,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.000000,0,0,1291,0,3.365220
7,extra_trees,test,0.030936,1291,0.0000,0.1549,0.998451,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.446512,0,2,1289,0,0.446377
13,lightgbm,test,0.000237,1291,0.0000,0.1549,0.998451,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.446512,0,2,1289,0,2.083157
3,logistic_regression,test,0.237423,1291,0.0000,0.3098,0.996902,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.893023,0,4,1287,0,0.176554
17,catboost,test,0.002934,1291,0.0000,10.3021,0.896979,NaN,0.000000,0.000000,0.000000,NaN,NaN,29.693023,0,133,1158,0,5.512634
11,mlp_neural_baseline,test,0.000487,1291,0.0000,18.2029,0.817971,NaN,0.000000,0.000000,0.000000,NaN,NaN,52.465116,0,235,1056,0,0.852458
1,dummy_prior,test,0.008801,1291,0.0000,100.0000,0.000000,NaN,0.000000,0.000000,0.000000,NaN,NaN,288.223256,0,1291,0,0,0.000815
8,hist_gradient_boosting,validation,0.005623,1291,0.9295,0.9295,0.998451,0.957942,0.916667,0.916667,0.916667,0.986057,0.914167,0.223256,11,1,1278,1,4.277012


## 9. Select Best Breach Classifier


In [ ]:
selection_frame = classification_results.loc[classification_results["split"].eq("validation")].copy()
selection_frame["pr_auc_for_sort"] = selection_frame["pr_auc"].fillna(-1)
selection_frame["recall_for_sort"] = selection_frame["recall"].fillna(0)
selection_frame["precision_for_sort"] = selection_frame["precision"].fillna(0)

best_classifier_row = selection_frame.sort_values(
    ["pr_auc_for_sort", "recall_for_sort", "precision_for_sort", "false_alarms_per_day"],
    ascending=[False, False, False, True],
).iloc[0]

best_classifier_name = best_classifier_row["model"]
best_classifier = classification_fitted[best_classifier_name]
best_classifier_threshold = classification_thresholds[best_classifier_name]

{
    "best_classifier": best_classifier_name,
    "chosen_threshold": best_classifier_threshold,
    "validation_pr_auc": best_classifier_row["pr_auc"],
    "validation_recall": best_classifier_row["recall"],
    "validation_precision": best_classifier_row["precision"],
}


# Robust selection remains validation-first, but later cells add walk-forward,
# incident-holdout, lead-time, and calibration evidence for the model card.
robust_candidate_scores = selection_frame[[
    "model", "pr_auc", "recall", "precision", "false_alarms_per_day"
]].copy()


## 9A. Walk-Forward Validation With 60-Minute Embargo

This tests the candidate breach models across multiple chronological folds. The 60-minute embargo prevents training rows immediately before a validation window from seeing labels whose future horizon overlaps that validation period.


In [ ]:
def build_walk_forward_folds(frame, n_folds=4, min_train_fraction=0.40, validation_fraction=0.12, embargo_minutes=60):
    ordered = frame.sort_values("timestamp").reset_index(drop=True)
    n_rows = len(ordered)
    min_train_end = int(n_rows * min_train_fraction)
    validation_size = max(int(n_rows * validation_fraction), 1)
    possible_starts = np.linspace(min_train_end, n_rows - validation_size, n_folds).astype(int)
    gap_steps = int(np.ceil(embargo_minutes / preprocessing_audit.get("sample_interval_minutes", 5)))

    folds = []
    for fold_number, validation_start in enumerate(possible_starts, start=1):
        validation_end = min(validation_start + validation_size, n_rows)
        train_end = max(validation_start - gap_steps, 0)
        train_idx = np.arange(0, train_end)
        validation_idx = np.arange(validation_start, validation_end)
        if len(train_idx) == 0 or len(validation_idx) == 0:
            continue
        folds.append({
            "fold": fold_number,
            "train_idx": train_idx,
            "validation_idx": validation_idx,
            "train_start": ordered.loc[train_idx[0], "timestamp"],
            "train_end": ordered.loc[train_idx[-1], "timestamp"],
            "validation_start": ordered.loc[validation_idx[0], "timestamp"],
            "validation_end": ordered.loc[validation_idx[-1], "timestamp"],
        })
    return ordered, folds


robust_model_names = [
    name for name in [
        "hist_gradient_boosting", "lightgbm", "xgboost",
        "random_forest", "extra_trees", "logistic_regression"
    ]
    if name in classification_models
]

wf_frame, wf_folds = build_walk_forward_folds(model_df, n_folds=4, min_train_fraction=0.40, validation_fraction=0.12, embargo_minutes=60)
walk_forward_rows = []

for model_name in robust_model_names:
    for fold in wf_folds:
        train_fold = wf_frame.iloc[fold["train_idx"]]
        valid_fold = wf_frame.iloc[fold["validation_idx"]]
        y_train_fold = train_fold[target].astype(int)
        y_valid_fold = valid_fold[target].astype(int)

        if y_train_fold.nunique() < 2:
            continue

        model = clone(classification_models[model_name])
        model.fit(train_fold[feature_cols], y_train_fold)

        if hasattr(model, "predict_proba"):
            score = model.predict_proba(valid_fold[feature_cols])[:, 1]
        else:
            raw = model.decision_function(valid_fold[feature_cols])
            score = (raw - raw.min()) / max(raw.max() - raw.min(), 1e-9)

        threshold, _ = choose_operational_threshold(
            valid_fold,
            y_valid_fold,
            score,
            min_recall=0.85,
            max_false_alarms_per_day=1.0,
        )
        metrics = evaluate_classifier(model_name, f"walk_forward_fold_{fold['fold']}", valid_fold, y_valid_fold, score, threshold)
        metrics.update({
            "fold": fold["fold"],
            "train_start": fold["train_start"],
            "train_end": fold["train_end"],
            "validation_start": fold["validation_start"],
            "validation_end": fold["validation_end"],
        })
        walk_forward_rows.append(metrics)

walk_forward_results = pd.DataFrame(walk_forward_rows)
walk_forward_summary = (
    walk_forward_results
    .groupby("model", dropna=False)
    .agg(
        folds=("fold", "nunique"),
        mean_pr_auc=("pr_auc", "mean"),
        mean_recall=("recall", "mean"),
        mean_precision=("precision", "mean"),
        mean_f1=("f1", "mean"),
        mean_false_alarms_per_day=("false_alarms_per_day", "mean"),
    )
    .reset_index()
    .sort_values(["mean_pr_auc", "mean_recall", "mean_false_alarms_per_day"], ascending=[False, False, True])
)
walk_forward_summary


## 9B. Incident-Focused Holdout Evaluation

This evaluates whole incident windows as mini holdouts. The incident label is audit context only and is not used as a predictor.


In [ ]:
def build_incident_holdouts(frame, pre_context_minutes=120, post_context_minutes=60, embargo_minutes=60):
    if "incident_group_id" not in frame.columns:
        return []
    incidents = (
        frame.loc[frame["incident_group_id"].ne("normal")]
        .groupby("incident_group_id", sort=False)
        .agg(
            incident_start=("timestamp", "min"),
            incident_end=("timestamp", "max"),
            event_type=("event_type", "first"),
            rows=("timestamp", "size"),
        )
        .reset_index()
        .sort_values("incident_start")
    )
    holdouts = []
    for _, row in incidents.iterrows():
        holdout_start = row["incident_start"] - pd.Timedelta(minutes=pre_context_minutes)
        holdout_end = row["incident_end"] + pd.Timedelta(minutes=post_context_minutes)
        train_cutoff = holdout_start - pd.Timedelta(minutes=embargo_minutes)
        train_mask = frame["timestamp"].lt(train_cutoff)
        eval_mask = frame["timestamp"].between(holdout_start, holdout_end)
        if train_mask.sum() and eval_mask.sum():
            holdouts.append({**row.to_dict(), "train_mask": train_mask, "eval_mask": eval_mask})
    return holdouts


incident_holdout_rows = []
incident_holdouts = build_incident_holdouts(model_df)

for model_name in robust_model_names:
    for holdout in incident_holdouts:
        train_fold = model_df.loc[holdout["train_mask"]]
        eval_fold = model_df.loc[holdout["eval_mask"]]
        y_train_fold = train_fold[target].astype(int)
        y_eval_fold = eval_fold[target].astype(int)
        if y_train_fold.nunique() < 2:
            continue

        internal_valid_start = int(len(train_fold) * 0.85)
        fit_fold = train_fold.iloc[:internal_valid_start]
        threshold_fold = train_fold.iloc[internal_valid_start:]
        y_fit = fit_fold[target].astype(int)
        y_threshold = threshold_fold[target].astype(int)

        if y_fit.nunique() < 2:
            continue

        model = clone(classification_models[model_name])
        model.fit(fit_fold[feature_cols], y_fit)

        if hasattr(model, "predict_proba"):
            threshold_score = model.predict_proba(threshold_fold[feature_cols])[:, 1]
            eval_score = model.predict_proba(eval_fold[feature_cols])[:, 1]
        else:
            threshold_raw = model.decision_function(threshold_fold[feature_cols])
            eval_raw = model.decision_function(eval_fold[feature_cols])
            threshold_score = (threshold_raw - threshold_raw.min()) / max(threshold_raw.max() - threshold_raw.min(), 1e-9)
            eval_score = (eval_raw - threshold_raw.min()) / max(threshold_raw.max() - threshold_raw.min(), 1e-9)

        if y_threshold.nunique() >= 2:
            threshold, _ = choose_operational_threshold(
                threshold_fold, y_threshold, threshold_score,
                min_recall=0.85, max_false_alarms_per_day=1.0
            )
        else:
            threshold = classification_thresholds.get(model_name, 0.5)

        metrics = evaluate_classifier(
            model_name,
            "incident_holdout",
            eval_fold,
            y_eval_fold,
            eval_score,
            threshold,
        )
        metrics.update({
            "incident_group_id": holdout["incident_group_id"],
            "event_type": holdout["event_type"],
            "incident_start": holdout["incident_start"],
            "incident_end": holdout["incident_end"],
        })
        incident_holdout_rows.append(metrics)

incident_holdout_results = pd.DataFrame(incident_holdout_rows)
incident_holdout_summary = (
    incident_holdout_results
    .groupby("model", dropna=False)
    .agg(
        holdouts=("incident_group_id", "nunique"),
        mean_pr_auc=("pr_auc", "mean"),
        mean_recall=("recall", "mean"),
        mean_precision=("precision", "mean"),
        mean_f1=("f1", "mean"),
        mean_false_alarms_per_day=("false_alarms_per_day", "mean"),
    )
    .reset_index()
    .sort_values(["mean_pr_auc", "mean_recall", "mean_false_alarms_per_day"], ascending=[False, False, True])
    if len(incident_holdout_results) else pd.DataFrame()
)
incident_holdout_summary


## 9C. Lead-Time and Calibration Checks


In [ ]:
best_valid_score = classification_scores[best_classifier_name]["validation"]
best_test_score = classification_scores[best_classifier_name]["test"]

lead_time_validation = breach_episode_lead_times(
    valid_df,
    best_valid_score,
    best_classifier_threshold,
    max_lookback_minutes=preprocessing_audit.get("max_prediction_horizon_minutes", 60),
)
lead_time_test = breach_episode_lead_times(
    test_df,
    best_test_score,
    best_classifier_threshold,
    max_lookback_minutes=preprocessing_audit.get("max_prediction_horizon_minutes", 60),
)

lead_time_results = pd.concat([
    lead_time_validation.assign(split="validation", model=best_classifier_name),
    lead_time_test.assign(split="test", model=best_classifier_name),
], ignore_index=True)

lead_time_summary = (
    lead_time_results
    .groupby(["model", "split"], dropna=False)
    .agg(
        breach_episodes=("breach_episode_id", "nunique"),
        caught_before_breach=("caught_before_breach", "sum"),
        average_lead_time_minutes=("lead_time_minutes", "mean"),
        median_lead_time_minutes=("lead_time_minutes", "median"),
    )
    .reset_index()
    if len(lead_time_results) else pd.DataFrame()
)

calibration_results = pd.concat([
    calibration_table(best_classifier_name, "validation", y_valid, best_valid_score),
    calibration_table(best_classifier_name, "test", y_test, best_test_score),
], ignore_index=True)

lead_time_summary, calibration_results.head()


## 10. Feature Importance for Best Breach Classifier


In [ ]:
def extract_tree_feature_importance(model, feature_names):
    fitted = model
    if isinstance(model, Pipeline):
        fitted = model.named_steps.get("model", model)
    if hasattr(fitted, "feature_importances_"):
        return pd.DataFrame({
            "feature": feature_names,
            "importance": fitted.feature_importances_,
            "importance_type": "native",
        }).sort_values("importance", ascending=False)
    if hasattr(fitted, "coef_"):
        coef = np.ravel(fitted.coef_)
        return pd.DataFrame({
            "feature": feature_names,
            "importance": np.abs(coef),
            "signed_coefficient": coef,
            "importance_type": "absolute_coefficient",
        }).sort_values("importance", ascending=False)
    return pd.DataFrame(columns=["feature", "importance", "importance_type"])


importance = extract_tree_feature_importance(best_classifier, feature_cols)

if importance.empty:
    sample_n = min(800, len(X_valid))
    perm = permutation_importance(
        best_classifier,
        X_valid.iloc[:sample_n],
        y_valid.iloc[:sample_n],
        scoring="average_precision",
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": perm.importances_mean,
        "importance_std": perm.importances_std,
        "importance_type": "permutation_average_precision",
    }).sort_values("importance", ascending=False)

importance["model"] = best_classifier_name
importance.head(25)


## 11. Forecasting Candidate Models


In [ ]:
forecast_models = {
    "dummy_mean": DummyRegressor(strategy="mean"),
    "ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0)),
    ]),
    "random_forest": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "extra_trees": ExtraTreesRegressor(
        n_estimators=350,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "hist_gradient_boosting": HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=350,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=0.05,
        random_state=RANDOM_STATE,
    ),
    "mlp_neural_baseline": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(
            hidden_layer_sizes=(96, 48),
            alpha=1e-3,
            learning_rate_init=1e-3,
            max_iter=500,
            early_stopping=True,
            random_state=RANDOM_STATE,
        )),
    ]),
}

if optional_libraries["lightgbm"]:
    forecast_models["lightgbm"] = LGBMRegressor(
        objective="regression",
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.9,
        colsample_bytree=0.85,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )

if optional_libraries["xgboost"]:
    forecast_models["xgboost"] = XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.85,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

if optional_libraries["catboost"]:
    forecast_models["catboost"] = CatBoostRegressor(
        iterations=500,
        learning_rate=0.03,
        depth=5,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False,
    )

list(forecast_models)


## 12. Train and Compare Forecasting Models


In [ ]:
def persistence_feature_for_target(target_name):
    mapping = {
        "target_cod_mg_l_plus_30min": "cod_mg_l_rt__imputed",
        "target_tss_mg_l_plus_30min": "tss_mg_l_rt__imputed",
        "target_bod_mg_l_plus_30min": "bod_mg_l_rt__imputed",
        "target_ammonia_mg_l_plus_30min": "ammonia_mg_l_rt__imputed",
        "target_ph_plus_30min": "ph_rt__imputed",
    }
    return mapping[target_name]


forecast_rows = []
forecast_fitted = {}

for forecast_target in forecast_target_cols:
    y_train_reg = train_df[forecast_target].astype(float)
    y_valid_reg = valid_df[forecast_target].astype(float)
    y_test_reg = test_df[forecast_target].astype(float)

    ok_train = y_train_reg.notna()
    ok_valid = y_valid_reg.notna()
    ok_test = y_test_reg.notna()

    persistence_col = persistence_feature_for_target(forecast_target)
    for split_name, frame, y_true, ok in [
        ("validation", valid_df, y_valid_reg, ok_valid),
        ("test", test_df, y_test_reg, ok_test),
    ]:
        y_pred = frame.loc[ok, persistence_col].to_numpy()
        forecast_rows.append(regression_metrics(
            "persistence_current_value",
            forecast_target,
            split_name,
            y_true.loc[ok].to_numpy(),
            y_pred,
        ))

    for model_name, estimator in forecast_models.items():
        model = clone(estimator)
        start = time.perf_counter()
        model.fit(X_train.loc[ok_train], y_train_reg.loc[ok_train])
        fit_seconds = time.perf_counter() - start

        valid_pred = model.predict(X_valid.loc[ok_valid])
        test_pred = model.predict(X_test.loc[ok_test])
        valid_metrics = regression_metrics(model_name, forecast_target, "validation", y_valid_reg.loc[ok_valid].to_numpy(), valid_pred)
        test_metrics = regression_metrics(model_name, forecast_target, "test", y_test_reg.loc[ok_test].to_numpy(), test_pred)
        valid_metrics["fit_seconds"] = fit_seconds
        test_metrics["fit_seconds"] = fit_seconds
        forecast_rows.extend([valid_metrics, test_metrics])
        forecast_fitted[(forecast_target, model_name)] = model

forecast_results = pd.DataFrame(forecast_rows).sort_values(["target", "split", "mae"])
forecast_results


## 13. Select Best Forecasting Models


In [ ]:
best_forecasters = {}
best_forecaster_rows = []

for forecast_target in forecast_target_cols:
    frame = forecast_results.loc[
        (forecast_results["target"].eq(forecast_target)) &
        (forecast_results["split"].eq("validation")) &
        (~forecast_results["model"].eq("persistence_current_value"))
    ].copy()
    best_row = frame.sort_values(["mae", "rmse"]).iloc[0]
    best_model_name = best_row["model"]
    best_forecasters[forecast_target] = forecast_fitted[(forecast_target, best_model_name)]
    best_forecaster_rows.append(best_row)

best_forecaster_summary = pd.DataFrame(best_forecaster_rows)
best_forecaster_summary


## 14. Anomaly Detection Layer


In [ ]:
anomaly_feature_candidates = [
    col for col in feature_cols
    if (
        "_jump_flag" in col
        or "missing_or_invalid" in col
        or "rolling_30min_slope" in col
        or "rolling_60min_std" in col
        or "min_margin_pct" in col
        or col in [
            "cod_mg_l_rt__imputed", "tss_mg_l_rt__imputed", "bod_mg_l_rt__imputed",
            "ammonia_mg_l_rt__imputed", "ph_rt__imputed", "flow_rate_lps_rt__imputed",
        ]
    )
]
anomaly_feature_candidates = list(dict.fromkeys(anomaly_feature_candidates))

normal_train = train_df["breach_now_bool"].astype(int).eq(0)
X_anom_train = train_df.loc[normal_train, anomaly_feature_candidates]
X_anom_valid = valid_df[anomaly_feature_candidates]
X_anom_test = test_df[anomaly_feature_candidates]

y_anom_valid = valid_df["breach_now_bool"].astype(int)
y_anom_test = test_df["breach_now_bool"].astype(int)

anomaly_models = {
    "rule_sensor_quality": None,
    "isolation_forest": IsolationForest(
        n_estimators=300,
        contamination=0.02,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "local_outlier_factor": LocalOutlierFactor(
        n_neighbors=35,
        contamination=0.02,
        novelty=True,
        n_jobs=-1,
    ),
    "one_class_svm": Pipeline([
        ("scaler", StandardScaler()),
        ("model", OneClassSVM(kernel="rbf", nu=0.02, gamma="scale")),
    ]),
}

anomaly_rows = []
anomaly_fitted = {}

for model_name, model in anomaly_models.items():
    if model_name == "rule_sensor_quality":
        valid_raw = (
            valid_df.get("any_sensor_jump_flag__imputed", 0)
            + valid_df.get("missing_or_invalid_count_60min__imputed", 0)
            + (valid_df.get("min_margin_pct_rt__imputed", 100) < 5).astype(int)
        )
        test_raw = (
            test_df.get("any_sensor_jump_flag__imputed", 0)
            + test_df.get("missing_or_invalid_count_60min__imputed", 0)
            + (test_df.get("min_margin_pct_rt__imputed", 100) < 5).astype(int)
        )
        valid_score = np.asarray(valid_raw, dtype=float)
        test_score = np.asarray(test_raw, dtype=float)
    else:
        fitted = clone(model)
        fitted.fit(X_anom_train)
        anomaly_fitted[model_name] = fitted
        if hasattr(fitted, "decision_function"):
            valid_normality = fitted.decision_function(X_anom_valid)
            test_normality = fitted.decision_function(X_anom_test)
            valid_score = -np.asarray(valid_normality)
            test_score = -np.asarray(test_normality)
        else:
            valid_pred = fitted.predict(X_anom_valid)
            test_pred = fitted.predict(X_anom_test)
            valid_score = (valid_pred == -1).astype(float)
            test_score = (test_pred == -1).astype(float)

    # Scale scores to 0-1 for thresholding.
    low, high = np.nanmin(valid_score), np.nanmax(valid_score)
    valid_scaled = (valid_score - low) / max(high - low, 1e-9)
    test_scaled = (test_score - low) / max(high - low, 1e-9)
    threshold, _ = choose_threshold(y_anom_valid, valid_scaled, min_recall=0.75)
    anomaly_rows.append(evaluate_classifier(model_name, "validation", valid_df, y_anom_valid, valid_scaled, threshold))
    anomaly_rows.append(evaluate_classifier(model_name, "test", test_df, y_anom_test, test_scaled, threshold))

anomaly_results = pd.DataFrame(anomaly_rows).sort_values(["split", "pr_auc", "recall"], ascending=[True, False, False])
anomaly_results


## 15. Select Best Anomaly Model


In [ ]:
anom_selection = anomaly_results.loc[anomaly_results["split"].eq("validation")].copy()
anom_selection["pr_auc_for_sort"] = anom_selection["pr_auc"].fillna(-1)
best_anomaly_row = anom_selection.sort_values(
    ["pr_auc_for_sort", "recall", "false_alarms_per_day"],
    ascending=[False, False, True],
).iloc[0]
best_anomaly_name = best_anomaly_row["model"]
best_anomaly_model = anomaly_fitted.get(best_anomaly_name)

{
    "best_anomaly_model": best_anomaly_name,
    "validation_pr_auc": best_anomaly_row["pr_auc"],
    "validation_recall": best_anomaly_row["recall"],
}


## 16. Save Results and Best Models


Artifact policy for this prototype:

- `.joblib` is the primary backend-serving artifact because the selected system is a Python bundle containing multiple scikit-learn/tree models, thresholds, feature lists, results, and metadata.
- `.h5` is also written as an HDF5-packaged copy of the same bundle for demo/project requirements. It is not a native Keras/deep-learning model.
- ONNX export is intentionally skipped for now because this is a multi-model bundle with mixed estimators and rule logic; add ONNX later only for individual supported estimators if the serving stack requires it.


In [ ]:
classification_results.to_csv(OUTPUTS["classification_results"], index=False)
forecast_results.to_csv(OUTPUTS["forecast_results"], index=False)
anomaly_results.to_csv(OUTPUTS["anomaly_results"], index=False)
importance.to_csv(OUTPUTS["feature_importance"], index=False)
walk_forward_results.to_csv(OUTPUTS["walk_forward_results"], index=False)
incident_holdout_results.to_csv(OUTPUTS["incident_holdout_results"], index=False)
lead_time_results.to_csv(OUTPUTS["lead_time_results"], index=False)
calibration_results.to_csv(OUTPUTS["calibration_results"], index=False)

model_bundle = {
    "metadata": {
        "created_by": "notebooks/04_model_training_evaluation.ipynb",
        "target": target,
        "feature_columns": feature_cols,
        "anomaly_feature_columns": anomaly_feature_candidates,
        "best_classifier_name": best_classifier_name,
        "best_classifier_threshold": float(best_classifier_threshold),
        "best_forecaster_names": {
            row["target"]: row["model"] for _, row in best_forecaster_summary.iterrows()
        },
        "best_anomaly_name": best_anomaly_name,
        "optional_libraries": optional_libraries,
        "walk_forward_summary": walk_forward_summary.to_dict(orient="records") if "walk_forward_summary" in globals() else [],
        "incident_holdout_summary": incident_holdout_summary.to_dict(orient="records") if "incident_holdout_summary" in globals() else [],
        "lead_time_summary": lead_time_summary.to_dict(orient="records") if "lead_time_summary" in globals() else [],
    },
    "breach_classifier": best_classifier,
    "forecast_models": best_forecasters,
    "anomaly_model": best_anomaly_model,
    "classification_results": classification_results,
    "forecast_results": forecast_results,
    "anomaly_results": anomaly_results,
    "feature_importance": importance,
}

joblib.dump(model_bundle, OUTPUTS["best_model_joblib"])

selection_summary = {
    "best_classifier": best_classifier_name,
    "best_classifier_threshold": float(best_classifier_threshold),
    "best_forecasters": {
        row["target"]: {
            "model": row["model"],
            "validation_mae": float(row["mae"]),
            "validation_rmse": float(row["rmse"]),
        }
        for _, row in best_forecaster_summary.iterrows()
    },
    "best_anomaly_model": best_anomaly_name,
    "validation_false_alarms_per_day": float(best_classifier_row["false_alarms_per_day"]),
    "validation_precision": float(best_classifier_row["precision"]),
    "validation_recall": float(best_classifier_row["recall"]),
    "validation_pr_auc": float(best_classifier_row["pr_auc"]) if pd.notna(best_classifier_row["pr_auc"]) else None,
    "walk_forward_summary": walk_forward_summary.to_dict(orient="records") if "walk_forward_summary" in globals() else [],
    "incident_holdout_summary": incident_holdout_summary.to_dict(orient="records") if "incident_holdout_summary" in globals() else [],
    "lead_time_summary": lead_time_summary.to_dict(orient="records") if "lead_time_summary" in globals() else [],
    "outputs": {name: str(path.relative_to(ROOT)) for name, path in OUTPUTS.items()},
    "h5_note": (
        "The .h5 file stores a pickle/joblib-compatible byte payload and metadata in HDF5. "
        "Use the .joblib artifact for normal sklearn loading."
    ),
}
OUTPUTS["model_selection"].write_text(json.dumps(selection_summary, indent=2, default=str), encoding="utf-8")

model_card = {
    "model_purpose": "Predict future wastewater compliance breach risk and support early warning alerts.",
    "regulatory_framing": "Configurable site-specific trade effluent consent / environmental permit limits, not universal legal thresholds.",
    "data_basis": "Simulated wastewater telemetry plus lab/soft-sensor artifacts in this prototype.",
    "selected_classifier": best_classifier_name,
    "selected_threshold": float(best_classifier_threshold),
    "feature_count": len(feature_cols),
    "known_limitations": [
        "Current final test period contains no positive breach windows.",
        "Production use requires facility-specific sensors, lab results, consent limits, incident records, and calibration logs.",
        "Model outputs support operator decisioning; deterministic rules remain authoritative for current breach status."
    ],
    "validation_summary": selection_summary,
}
OUTPUTS["model_card"].write_text(json.dumps(model_card, indent=2, default=str), encoding="utf-8")

if optional_libraries["h5py"]:
    payload = pickle.dumps(model_bundle)
    with h5py.File(OUTPUTS["best_model_h5"], "w") as h5:
        h5.create_dataset("model_bundle_pickle", data=np.void(payload))
        meta = h5.create_group("metadata")
        for key, value in model_bundle["metadata"].items():
            meta.attrs[key] = json.dumps(value, default=str)
else:
    print("h5py is not installed, so the .h5 export was skipped. Install h5py if strict .h5 output is required.")

pd.DataFrame({
    "artifact": OUTPUTS.keys(),
    "path": [str(path.relative_to(ROOT)) for path in OUTPUTS.values()],
    "exists": [path.exists() for path in OUTPUTS.values()],
})


## 17. Final Comparison Tables


In [ ]:
display(classification_results.sort_values(["split", "pr_auc", "recall"], ascending=[True, False, False]))
display(threshold_comparison.sort_values(["model", "threshold"]))
display(forecast_results.sort_values(["target", "split", "mae"]))
display(anomaly_results.sort_values(["split", "pr_auc", "recall"], ascending=[True, False, False]))
display(importance.head(30))

display(walk_forward_summary)
display(incident_holdout_summary)
display(lead_time_summary)
display(calibration_results.head(20))

## 18. Notes for Interpretation

The current chronological test split has no positive `target_breach_next_30min` rows. That is useful as a false-alarm stress test, but not enough for final breach recall.

Use the validation split and, if needed, a later walk-forward/embargoed validation design for breach-recall claims. Keep the final test report honest:

```text
test recall is undefined when the test period has no positive breach windows
test false alarms/day is still meaningful
```

For the pitch, report:

- best validation PR-AUC and recall for breach prediction;
- false alarms per day at the chosen threshold;
- COD/TSS/BOD/ammonia/pH forecast MAE vs persistence;
- anomaly layer PR-AUC/recall against current breach rows;
- top interpretable drivers from feature importance.


Production-minded interpretation:

- use walk-forward validation to estimate breach recall across time;
- use incident-focused holdouts to test whole-event generalization;
- use threshold tuning constrained by false alarms/day;
- use lead-time metrics to prove proactive warning, not just classification accuracy;
- keep deterministic rules as the authority for current compliance status.
